In [ ]:
# Base code for lab 5 in-lab task solution. Now carried over to DAG

In [17]:
# ei tea kas see päriselt vajalik, lihtsalt labist võetud
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "psycopg2-binary", "kafka-python-ng"])
print("requests + psycopg2 + kafka-python-ng ready")

requests + psycopg2 + kafka-python-ng ready


In [1]:
# Start of task 2
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import os

spark = (SparkSession.builder 
  .appName("CDC-Bronze") 
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
  .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") 
  .config("spark.sql.catalog.lakehouse.type", "rest") 
  .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
  .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")  
  .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
  .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
  .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
  .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
  .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
  .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
  .config("spark.sql.defaultCatalog", "lakehouse") 
  .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}")

Spark 4.1.0


In [19]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")

DataFrame[]

In [20]:
raw = (spark.read
  .format("kafka") 
  .option("kafka.bootstrap.servers", "kafka:9092") 
  .option("subscribe", "dbserver1.public.customers") 
  .option("startingOffsets", "earliest") 
  .load())

In [21]:
from pyspark.sql import functions as F


# Filter out tombstone records (null value) first
raw_filtered = raw.filter(F.col("value").isNotNull())


bronze_df = raw_filtered.select(
  F.col("topic"),
  F.col("partition").alias("kafka_partition"),
  F.col("offset").alias("kafka_offset"),
  F.col("timestamp").alias("kafka_timestamp"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("long").alias("ts_ms"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.name").alias("after_name"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.email").alias("after_email"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.country").alias("after_country"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
)

In [29]:
bronze_df.writeTo("lakehouse.cdc.bronze_customers").createOrReplace() # change to append() in actual project

In [30]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.bronze_customers").show()

spark.sql("""

  SELECT op, after_id, after_name, after_email, ts_ms

  FROM lakehouse.cdc.bronze_customers LIMIT 5

""").show(truncate=False)

+--------+
|count(1)|
+--------+
|     426|
+--------+

+---+--------+--------------+-----------------+-------------+
|op |after_id|after_name    |after_email      |ts_ms        |
+---+--------+--------------+-----------------+-------------+
|r  |2       |Bob Virtanen  |bob@example.com  |1776346913293|
|r  |3       |Carol Ozols   |carol@example.com|1776346913295|
|r  |4       |David Jonaitis|david@example.com|1776346913295|
|r  |6       |Frank Muller  |frank@example.com|1776346913296|
|r  |7       |Grace Kim     |grace@example.com|1776346913296|
+---+--------+--------------+-----------------+-------------+



In [31]:
# Task 3

spark.sql("""

  CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_customers (

    id INT, name STRING, email STRING, country STRING, last_updated_ms BIGINT

  ) USING iceberg

""")

DataFrame[]

In [32]:
from pyspark.sql.window import Window


# Use COALESCE because for deletes, after_id is null but before_id has the key

bronze_with_key = bronze_df.withColumn(

  "entity_id", F.coalesce(F.col("after_id"), F.col("before_id"))

)


w = Window.partitionBy("entity_id").orderBy(F.col("ts_ms").desc())

deduped = bronze_with_key.filter(F.col("op").isNotNull()).withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

In [33]:
deduped.createOrReplaceTempView("cdc_batch")


spark.sql("""

  MERGE INTO lakehouse.cdc.silver_customers AS t

  USING cdc_batch AS s

  ON t.id = s.entity_id


  WHEN MATCHED AND s.op = 'd' THEN DELETE


  WHEN MATCHED AND s.op IN ('c','u','r') THEN UPDATE SET

    t.name = s.after_name, t.email = s.after_email,

    t.country = s.after_country, t.last_updated_ms = s.ts_ms


  WHEN NOT MATCHED AND s.op != 'd' THEN INSERT

    (id, name, email, country, last_updated_ms)

    VALUES (s.after_id, s.after_name, s.after_email, s.after_country, s.ts_ms)

""")

DataFrame[]

In [34]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_customers").show()

spark.sql("SELECT * FROM lakehouse.cdc.silver_customers ORDER BY id LIMIT 5").show(truncate=False)

+--------+
|count(1)|
+--------+
|     111|
+--------+

+---+--------------+-------------------------+-----------+---------------+
|id |name          |email                    |country    |last_updated_ms|
+---+--------------+-------------------------+-----------+---------------+
|1  |Alice Mets    |updated_1_501@mail.com   |Lithuania  |1776347498224  |
|2  |Bob Virtanen  |updated_2_415@mail.com   |Netherlands|1776347077838  |
|4  |David Jonaitis|updated_4_841@example.com|Lithuania  |1776347021122  |
|37 |Olivia Tanaka |updated_37_108@inbox.org |India      |1776347468656  |
|38 |Diego Muller  |updated_38_776@inbox.org |Norway     |1776347249197  |
+---+--------------+-------------------------+-----------+---------------+



In [2]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT count(*) FROM lakehouse.taxi.silver_trips").show()

+--------+
|count(1)|
+--------+
|     198|
+--------+

+--------+
|count(1)|
+--------+
| 5478063|
+--------+



In [20]:
spark.sql("SELECT * FROM lakehouse.taxi.silver_trips LIMIT 5").show()
spark.sql("SELECT * FROM lakehouse.cdc.silver_customers LIMIT 5").show()
# Check if drivers table exists yet
spark.sql("SHOW TABLES IN lakehouse.cdc").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|            

In [8]:
# Step 1: trips per zone per day per hour

spark.sql("""
    CREATE OR REPLACE TEMP VIEW hourly_zone_trips AS
    SELECT
        PULocationID AS zone_id,
        date(tpep_pickup_datetime) AS trip_date,
        hour(tpep_pickup_datetime) AS hour_of_day,
        count(*) AS trip_count
    FROM lakehouse.taxi.silver_trips
    GROUP BY PULocationID, date(tpep_pickup_datetime), hour(tpep_pickup_datetime)
""")

DataFrame[]

In [10]:
# Step 2: aggregate across days to get avg and stddev per zone+hour
spark.sql("""
    CREATE OR REPLACE TEMP VIEW demand_stats AS
    SELECT
        zone_id,
        hour_of_day,
        avg(trip_count) AS avg_trips,
        stddev(trip_count) AS stddev_trips
    FROM hourly_zone_trips
    GROUP BY zone_id, hour_of_day
""")

DataFrame[]

In [11]:
# Step 3: classify demand
spark.sql("""
    SELECT
        zone_id,
        hour_of_day,
        avg_trips,
        stddev_trips,
        CASE
            WHEN avg_trips > (SELECT avg(avg_trips) + stddev(avg_trips) FROM demand_stats) THEN 'high demand'
            WHEN avg_trips < (SELECT avg(avg_trips) - stddev(avg_trips) FROM demand_stats) THEN 'low demand'
            ELSE 'normal'
        END AS demand_classification
    FROM demand_stats
    ORDER BY zone_id, hour_of_day
""").show()

+-------+-----------+------------------+-------------------+---------------------+
|zone_id|hour_of_day|         avg_trips|       stddev_trips|demand_classification|
+-------+-----------+------------------+-------------------+---------------------+
|      1|          4|               1.0|               NULL|               normal|
|      1|          5|               1.0|                0.0|               normal|
|      1|          6|               1.0|                0.0|               normal|
|      1|          7|               1.0|                0.0|               normal|
|      1|          9|               1.0|                0.0|               normal|
|      1|         10|               1.0|                0.0|               normal|
|      1|         12|               1.5|  1.224744871391589|               normal|
|      1|         13|1.1818181818181819|0.40451991747794525|               normal|
|      1|         14|1.0833333333333333| 0.2886751345948129|               normal|
|   

In [18]:
# Which 3 zones have most predictable demand (lowest stddev)?
spark.sql("""
    SELECT zone_id, avg(coalesce(stddev_trips, 0)) AS avg_stddev
    FROM demand_stats
    GROUP BY zone_id
    ORDER BY avg_stddev ASC
    LIMIT 3
""").show()

# At what hour does demand peak city-wide?
spark.sql("""
    SELECT hour_of_day, sum(avg_trips) AS total_avg_trips
    FROM demand_stats
    GROUP BY hour_of_day
    ORDER BY total_avg_trips DESC
    LIMIT 3
""").show()

+-------+----------+
|zone_id|avg_stddev|
+-------+----------+
|    115|       0.0|
|     31|       0.0|
|     27|       0.0|
+-------+----------+

+-----------+-----------------+
|hour_of_day|  total_avg_trips|
+-----------+-----------------+
|         18|6962.702695040019|
|         17|6896.985896392369|
|         16|6339.490257920653|
+-----------+-----------------+



In [19]:
spark.sql("SELECT * FROM lakehouse.cdc.bronze_customers LIMIT 1").show(truncate=False)

+--------------------------+---------------+------------+-----------------------+---+-------------+--------+----------+-----------------+-------------+---------+
|topic                     |kafka_partition|kafka_offset|kafka_timestamp        |op |ts_ms        |after_id|after_name|after_email      |after_country|before_id|
+--------------------------+---------------+------------+-----------------------+---+-------------+--------+----------+-----------------+-------------+---------+
|dbserver1.public.customers|0              |0           |2026-05-02 08:42:54.851|r  |1777711374591|1       |Alice Mets|alice@example.com|Estonia      |NULL     |
+--------------------------+---------------+------------+-----------------------+---+-------------+--------+----------+-----------------+-------------+---------+

